In [1]:
import pandas as pd
import numpy as np
import re
import os

def clean_price(value):
    """清洗价格数据"""
    if pd.isna(value):
        return np.nan
    
    if isinstance(value, (int, float)):
        return float(value)
    
    value_str = str(value).strip()
    value_str = value_str.replace('\n', '').replace(' ', '')
    value_str = value_str.replace('¥', '').replace('￥', '')
    value_str = value_str.replace(',', '')
    
    if not value_str:
        return np.nan
    
    try:
        return float(value_str)
    except ValueError:
        cleaned = re.sub(r'[^\d.-]', '', value_str)
        if cleaned:
            try:
                return float(cleaned)
            except ValueError:
                return np.nan
        else:
            return np.nan

def get_price_threshold(file_name):
    """根据文件名获取价格阈值"""
    file_lower = file_name.lower()
    
    if '优思明' in file_lower:
        return 106.0
    elif '优思悦' in file_lower:
        return 151.0
    elif '唯散宁' in file_lower:
        return 459.0
    else:
        print(f"警告: 文件名中未识别到药品名称，使用默认阈值100")
        return 100.0

def get_drug_name(file_name):
    """从文件名中提取药品名称"""
    file_lower = file_name.lower()
    
    if '优思明' in file_lower:
        return '优思明'
    elif '优思悦' in file_lower:
        return '优思悦'
    elif '唯散宁' in file_lower:
        return '唯散宁'
    else:
        return '未知药品'

# 定义输入文件名
input_file = "D:\work\京东优思明0417.xlsx"

# 获取药品名称和价格阈值
drug_name = get_drug_name(input_file)
threshold = get_price_threshold(input_file)
print(f"检测到药品: {drug_name}, 价格阈值: {threshold}元")

# 从输入文件名生成输出文件名
file_name, file_ext = os.path.splitext(input_file)
# 完整处理结果文件
complete_output_file = f"{file_name}_去重后_无海外_sorted{file_ext}"
# 筛选结果文件
filtered_output_file = f"{file_name}_筛选低于{int(threshold)}元{file_ext}"

# 读取Excel文件
df = pd.read_excel(input_file)

print(f"原始数据文件: {input_file}")
print(f"完整处理结果文件: {complete_output_file}")
print(f"筛选结果文件: {filtered_output_file}")
print(f"筛选阈值: {threshold}元")

# 检查必要列是否存在
required_columns = ['到手_优惠价']
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print(f"\n错误：缺少必要的列: {missing_columns}")
    print(f"可用的列名: {df.columns.tolist()}")
else:
    print("\n开始处理数据...")
    
    # 记录原始行数
    original_count = len(df)
    
    # 1. 商品链接去重
    link_column = '商品链接'
    
    if link_column in df.columns:
        print(f"\n1. 商品链接去重处理...")
        duplicate_counts = df[link_column].value_counts()
        duplicate_links = duplicate_counts[duplicate_counts > 1]
        
        if not duplicate_links.empty:
            print(f"   发现 {len(duplicate_links)} 个重复商品链接")
            print(f"   重复数据行数: {duplicate_links.sum() - len(duplicate_links)} 行")
            
            # 显示前3个重复链接示例
            print(f"   重复链接示例（前3个）:")
            for link, count in duplicate_links.head(3).items():
                link_str = str(link)
                if len(link_str) > 50:
                    link_str = link_str[:47] + "..."
                print(f"     {link_str} : {count}次")
        
        df = df.drop_duplicates(subset=link_column, keep='first')
        removed_count = original_count - len(df)
        print(f"   已移除 {removed_count} 个重复商品链接")
        print(f"   去重后行数: {len(df)}")
    else:
        print(f"\n警告: 未找到'{link_column}'列，跳过商品链接去重步骤")
    
    # 2. 删除海外店铺
    shop_column = '店铺名称'
    
    if shop_column in df.columns:
        print(f"\n2. 删除海外店铺处理...")
        df_before_overseas = len(df)
        overseas_mask = df[shop_column].astype(str).str.contains('海外', case=False, na=False)
        overseas_count = overseas_mask.sum()
        
        if overseas_count > 0:
            print(f"   发现 {overseas_count} 行店铺名称包含'海外'的数据")
            
            # 显示前3个海外店铺示例
            overseas_shops = df[overseas_mask][shop_column].head(3)
            for i, shop in enumerate(overseas_shops, 1):
                print(f"   {i}. {shop}")
        
        df = df[~overseas_mask]
        removed_overseas_count = df_before_overseas - len(df)
        print(f"   已移除 {removed_overseas_count} 行包含'海外'的店铺数据")
        print(f"   删除后行数: {len(df)}")
    else:
        print(f"\n警告: 未找到'{shop_column}'列，跳过删除海外店铺步骤")
    
    # 3. 价格清洗
    print(f"\n3. 价格清洗处理...")
    df['到手_优惠价_清洗后'] = df['到手_优惠价'].apply(clean_price)
    
    # 统计清洗结果
    success_count = df['到手_优惠价_清洗后'].notna().sum()
    fail_count = df['到手_优惠价_清洗后'].isna().sum()
    print(f"   成功转换: {success_count} 行")
    print(f"   转换失败: {fail_count} 行")
    
    if fail_count > 0:
        print(f"   转换失败的行（前3行）：")
        nan_mask = df['到手_优惠价_清洗后'].isna()
        problematic = df[nan_mask][['到手_优惠价']].head(3)
        for i, (idx, row) in enumerate(problematic.iterrows(), 1):
            print(f"     {i}. 行{idx}: {repr(row['到手_优惠价'])}")
    
    # 4. 保存完整处理结果
    print(f"\n4. 保存完整处理结果...")
    # 按价格排序，NaN值放在最后
    df_sorted_all = df.sort_values('到手_优惠价_清洗后', ascending=True, na_position='last')
    df_sorted_all.to_excel(complete_output_file, index=False)
    print(f"   完整处理数据已保存: {complete_output_file}")
    
    # 显示完整数据的价格统计
    price_stats = df_sorted_all['到手_优惠价_清洗后'].describe()
    print(f"\n完整数据价格统计:")
    print(f"   行数: {len(df_sorted_all)}")
    print(f"   最小值: {price_stats['min']:.2f} 元")
    print(f"   最大值: {price_stats['max']:.2f} 元")
    print(f"   平均值: {price_stats['mean']:.2f} 元")
    print(f"   中位数: {price_stats['50%']:.2f} 元")
    
    # 5. 筛选低于阈值的数据
    print(f"\n5. 筛选低于阈值的数据（< {threshold}元）...")
    # 先删除清洗失败的行
    df_clean = df.dropna(subset=['到手_优惠价_清洗后'])
    print(f"   清洗后可用数据行数: {len(df_clean)}")
    
    # 筛选低于阈值的数据
    filtered_df = df_clean[df_clean['到手_优惠价_清洗后'] < threshold].copy()
    
    print(f"   筛选到 {len(filtered_df)} 行数据低于 {threshold} 元")
    
    if not filtered_df.empty:
        # 按价格排序
        filtered_df_sorted = filtered_df.sort_values('到手_优惠价_清洗后', ascending=True)
        
        # 价格统计
        min_price = filtered_df_sorted['到手_优惠价_清洗后'].min()
        max_price = filtered_df_sorted['到手_优惠价_清洗后'].max()
        avg_price = filtered_df_sorted['到手_优惠价_清洗后'].mean()
        
        print(f"\n筛选结果价格统计:")
        print(f"   最低价格: {min_price:.2f} 元")
        print(f"   最高价格: {max_price:.2f} 元")
        print(f"   平均价格: {avg_price:.2f} 元")
        
        # 显示前5个最低价格
        print(f"\n最低价格前5行:")
        for i, (idx, row) in enumerate(filtered_df_sorted.head(5).iterrows(), 1):
            shop_name = row.get('店铺名称', '未知') if '店铺名称' in row else '未知'
            price = row['到手_优惠价_清洗后']
            print(f"   {i}. 店铺: {shop_name[:20]:<20} 价格: {price:>7.2f} 元")
        
        # 6. 保存筛选结果
        print(f"\n6. 保存筛选结果...")
        filtered_df_sorted.to_excel(filtered_output_file, index=False)
        print(f"   筛选结果已保存: {filtered_output_file}")
        
        # 显示文件大小
        if os.path.exists(filtered_output_file):
            file_size = os.path.getsize(filtered_output_file) / 1024
            print(f"   筛选结果文件大小: {file_size:.1f} KB")
        
        # 显示筛选数据维度
        print(f"\n数据维度:")
        print(f"   原始数据: {original_count} 行 × {df.shape[1]} 列")
        print(f"   完整处理结果: {df_sorted_all.shape[0]} 行 × {df_sorted_all.shape[1]} 列")
        print(f"   筛选结果: {filtered_df_sorted.shape[0]} 行 × {filtered_df_sorted.shape[1]} 列")
        
    else:
        print(f"\n⚠️ 警告: 没有找到低于阈值 {threshold} 元的数据！")
        print("可能原因:")
        print(f"   1. 所有价格都高于 {threshold} 元")
        print("   2. 价格清洗可能有问题")
        print("   3. 阈值设置可能过低")
        
        # 创建一个空的筛选结果文件
        empty_df = pd.DataFrame(columns=df.columns)
        empty_df.to_excel(filtered_output_file, index=False)
        print(f"   创建空的筛选结果文件: {filtered_output_file}")

print(f"\n{'='*50}")
print("处理完成！")
print(f"1. 完整处理结果: {complete_output_file}")
print(f"2. 筛选结果: {filtered_output_file}")
print(f"{'='*50}")

检测到药品: 优思明, 价格阈值: 106.0元
原始数据文件: D:\work\京东优思明0417.xlsx
完整处理结果文件: D:\work\京东优思明0417_去重后_无海外_sorted.xlsx
筛选结果文件: D:\work\京东优思明0417_筛选低于106元.xlsx
筛选阈值: 106.0元

开始处理数据...

1. 商品链接去重处理...
   发现 39 个重复商品链接
   重复数据行数: 39 行
   重复链接示例（前3个）:
     https://item.jd.com/10217752215009.html : 2次
     https://item.jd.com/10217664056068.html : 2次
     https://item.jd.com/10217529130670.html : 2次
   已移除 39 个重复商品链接
   去重后行数: 1659

2. 删除海外店铺处理...
   已移除 0 行包含'海外'的店铺数据
   删除后行数: 1659

3. 价格清洗处理...
   成功转换: 1659 行
   转换失败: 0 行

4. 保存完整处理结果...
   完整处理数据已保存: D:\work\京东优思明0417_去重后_无海外_sorted.xlsx

完整数据价格统计:
   行数: 1659
   最小值: 102.50 元
   最大值: 29250.00 元
   平均值: 384.65 元
   中位数: 280.00 元

5. 筛选低于阈值的数据（< 106.0元）...
   清洗后可用数据行数: 1659
   筛选到 8 行数据低于 106.0 元

筛选结果价格统计:
   最低价格: 102.50 元
   最高价格: 105.80 元
   平均价格: 104.84 元

最低价格前5行:
   1. 店铺: 一品大药房官方旗舰店           价格:  102.50 元
   2. 店铺: 丰原大药房旗舰店             价格:  104.40 元
   3. 店铺: 昌盛大药房官方旗舰店           价格:  105.00 元
   4. 店铺: 国大(广西)大药房旗舰店         价格:  105.00 元
   

In [2]:
import pandas as pd
import os

def filter_ka_stores(input_file_path, sheet_name, candidate_stores):
    """
    筛选包含候选店铺名的数据行
    
    参数:
    input_file_path: 输入的Excel文件路径
    sheet_name: 要读取的Sheet页名称
    candidate_stores: 候选店铺列表
    """
    
    # 读取Excel文件指定的Sheet页
    try:
        df = pd.read_excel(input_file_path, sheet_name=sheet_name)
        print(f"成功读取Sheet页: {sheet_name}")
    except Exception as e:
        print(f"读取文件失败: {e}")
        
        # 显示可用的sheet页
        try:
            xl = pd.ExcelFile(input_file_path)
            sheet_names = xl.sheet_names
            print(f"可用的Sheet页: {sheet_names}")
        except:
            pass
        return
    
    # 检查必要的列是否存在 - 修改为'店铺名称'
    if '店铺名称' not in df.columns:
        print("错误: Excel文件中没有找到'店铺名称'列")
        print(f"可用列: {list(df.columns)}")
        return
    
    print(f"原始数据行数: {len(df)}")
    
    # 创建筛选条件：店铺名称列包含任一候选店铺名
    condition = pd.Series(False, index=df.index)
    
    for store in candidate_stores:
        # 检查店铺名称列是否包含候选店铺名（不区分大小写）
        condition = condition | df['店铺名称'].astype(str).str.contains(store, case=False, na=False)
    
    # 应用筛选条件
    filtered_df = df[condition]
    
    print(f"筛选后数据行数: {len(filtered_df)}")
    
    # 如果没有筛选到数据
    if len(filtered_df) == 0:
        print("\n警告: 没有筛选到任何数据！")
        print("可能的原因:")
        print("1. 店铺名称列中不包含任何候选店铺名")
        print("2. 候选店铺名与Excel中的店铺名称不匹配")
        
        # 显示前10个不同的店铺名称
        print(f"\n前10个不同的店铺名称:")
        unique_stores = df['店铺名称'].dropna().unique()[:10]
        for i, store_name in enumerate(unique_stores, 1):
            print(f"  {i}. {store_name}")
        
        # 询问是否继续保存空文件
        user_input = input("\n是否继续保存空文件? (y/n): ").strip().lower()
        if user_input != 'y':
            print("操作已取消")
            return None
    
    # 按日期和店铺排序（如果存在日期列）
    if '日期' in filtered_df.columns:
        print("按日期排序...")
        try:
            # 尝试转换为日期类型
            filtered_df['日期'] = pd.to_datetime(filtered_df['日期'])
            filtered_df = filtered_df.sort_values(['日期', '店铺名称'])
            print("已按日期排序")
        except:
            filtered_df = filtered_df.sort_values(['日期', '店铺名称'])
            print("已按日期排序（原始格式）")
    elif '日期' in df.columns and '日期' not in filtered_df.columns:
        # 如果原始数据有日期列但筛选后没有，添加回日期列
        if '日期' in df.columns:
            filtered_df = filtered_df.merge(df[['店铺名称', '日期']], on='店铺名称', how='left')
            filtered_df = filtered_df.sort_values(['日期', '店铺名称'])
            print("已添加并排序日期列")
    
    # 生成输出文件名
    file_dir = os.path.dirname(input_file_path)
    file_name = os.path.basename(input_file_path)
    name_part, ext = os.path.splitext(file_name)
    
    # 在文件名中包含sheet名
    output_file_name = f"{name_part}_{sheet_name}_KA{ext}"
    output_file_path = os.path.join(file_dir, output_file_name)
    
    # 保存筛选后的数据到新Excel文件
    try:
        filtered_df.to_excel(output_file_path, index=False)
        print(f"筛选完成！结果已保存到: {output_file_path}")
        
        # 显示保存的详细信息
        if len(filtered_df) > 0:
            print(f"\n筛选结果信息:")
            print(f"  总行数: {len(filtered_df)}")
            print(f"  总列数: {len(filtered_df.columns)}")
            
            # 按候选店铺统计数量
            print(f"\n按候选店铺统计匹配数量:")
            for store in candidate_stores:
                count = filtered_df['店铺名称'].astype(str).str.contains(store, case=False, na=False).sum()
                if count > 0:
                    print(f"  {store}: {count} 行")
        
        return output_file_path
    except Exception as e:
        print(f"保存文件失败: {e}")
        return None

def main():
    # 候选店铺列表
    candidate_stores = [
    "养和堂", "国大", "华氏", "好药师", "崇明第一医药", "第一医药商店", "得一大药房", "海王星辰", 
    "益丰", "雷允上", "一心堂", "老百姓", "东飞药业", "人民大药房", "为了你健康药房", "昊邦", 
    "白药大药房", "百姓仁药业", "医药新特药", "玉溪医药", "立之康", "靓桐", "健之佳", "淞茂", 
    "滇西", "益尔健", "得云药业", "新特新", "仁和堂", "北域健康", "厚道医药", "康中福", "德医堂", 
    "方圆", "惠生", "泽强", "达明堂", "东升天保堂", "日月七天", "金象", "东辽医药", "福春永", 
    "保康", "江城大药房", "福聚康", "敖东", "一笑堂", "万家星火", "义立方", "光明医药", "养天和康赛", 
    "华安大药房", "同仁大药房", "天诺", "恒爱", "房联", "新药特药", "春萍", "永新堂", "百姓润", 
    "百年红星", "禹成", "邱健安康", "金牛大药房", "骅禧", "神农", "益和", "中联保健", "天一堂", 
    "正道", "时珍", "恒爱", "光明万家", "万和堂", "信康", "汇丰", "长新", "义善堂", "三好名典", 
    "安顺堂", "好医生", "嘉宝堂正红", "东升", "健之佳福利", "全泰堂", "华鼎", "圣杰", "天天康", 
    "太极", "杏林", "正和祥", "万盛", "华安堂", "巴中怡和", "百欣堂", "芙蓉", "贝尔康", "高济", 
    "齐力堂", "龙一", "大参林", "广恩堂", "心健", "寿亨堂", "泰和", "芝心", "大成家人", "普济文军", 
    "万华", "健尔堂", "宁丰堂", "东升", "乐源堂", "华杏", "成毅康缘", "本草堂", "泉源堂", "百信", 
    "科伦", "敬仁堂", "东升", "国健", "眉州药房", "安康", "众益堂", "宇豪", "天泰同安", "卫康", 
    "京华壹零贰", "瑞澄", "德立信", "百姓乐", "惠仁堂", "美合泰", "丰原", "佳百姓", "华康", "四方百信", 
    "国春堂", "国胜", "天星", "尚本堂", "广济", "百十信", "邻加医康复", "立方", "达嘉维康", "韵天", 
    "高济敬贤堂", "鸿兴", "人民大药房", "仁和", "养生堂", "淮金益寿堂", "天平", "佳佳恒康", "新诚", 
    "华巨百姓缘", "庆瑞祥堂", "博利", "中山", "元初", "易安堂", "赵泰和堂", "江南", "天地仁和", 
    "杨博", "第一", "博爱", "曼迪新", "华仁堂", "同春", "花园", "康美", "仁德", "仁和堂", "立健", 
    "信宏仁", "国民医保", "德信堂", "民泰", "燕喜堂", "百年康润", "益寿堂", "联众", "葆春堂", "誉天", 
    "幸福人", "平民", "润康广场", "润生堂", "天天好", "百姓药业", "天和堂", "国康", "平嘉", "方成", 
    "广联", "仁合堂", "天和堂景泰成", "众康", "漱玉", "德信行", "金通", "万民阳光", "中医世家", 
    "惠仁", "健民", "五洲", "益民", "方正", "医保城", "同方", "国风", "紫光", "东方", "益源", 
    "竹林", "仁人和", "同人众鑫", "同仁康", "国大万民", "国邦", "思迈乐", "百姓药房", "百汇", 
    "荣华", "吉达惠康", "天诚", "新长城", "亚宝", "滋生堂", "永康堂", "中智", "佛心", "百姓堂", 
    "顺德", "北京同仁堂", "德信行", "国控国大", "健丰", "大源堂", "五福堂", "健药", "品健", 
    "好万家", "家之和", "恒青", "新合健", "春天一百", "深华", "爱心", "百源堂", "益康", "东莞国药", 
    "邦健", "东兴堂", "健民", "宝芝林", "林芝参", "正和", "民信", "集和堂", "广药大药房","广药", "康泽", 
    "卫康", "百姓缘", "立丰", "民德", "二天堂", "万泽", "中港", "南北药行", "民心", "瑞草堂", 
    "北湖", "益民", "健春堂", "一朝心阳", "新兴堂", "宝和堂", "康全济惠", "康全", "桂中", 
    "百姓人家", "梧州百姓", "康是美", "康之源", "安康", "惠生堂", "新特", "塔城", "新特药", 
    "库尔勒", "一心康达", "万佳康", "乐盈众康", "好又多", "康宁", "康泰东方", "普济堂", "汇泰", 
    "济康", "百草堂", "维之康", "达康", "绿珠", "腾龙", "礼安", "上元堂", "健桥", "天颐", "百信", 
    "恒玖信", "聚力堂", "天天乐", "京典", "江海", "普泽", "济生堂", "颐丹堂", "永泰", "大德生", 
    "天健", "一盛", "万仁", "中诚", "人寿天", "金坛百姓", "恒心远", "建发", "恒泰", "百禾", 
    "中健", "保庆堂", "广济", "恩华统一", "汉邦", "泰隆", "恒泰人民", "众成堂", "同康泰", "三品堂", 
    "九州", "山禾", "天禾堂", "汇华强盛", "烨辉", "双鹤同德堂", "万家春", "同慈", "万家康", "万泽", 
    "九鼎", "仁源生", "仁爱", "众佳康", "众药师", "健康家园", "华鑫源", "卫民", "同正", "启扬", 
    "嘉伦光彩", "国瑞堂", "大众", "大德隆", "奇冠", "市民", "康济", "海邦", "海鹏", "润天", 
    "百佳惠瑞丰", "百佳惠苏禾", "盐淮百信", "竹宸国瑞堂", "筑康", "联环健康", "舒健", "赛邦", 
    "兴药", "为你想", "平民", "张氏", "德新元", "芝林", "洪医", "济生", "广济", "顺泰", "先锋", 
    "正泰", "东方红", "百草堂", "健生源", "天一", "采芝灵", "开开心心", "粤海永熙堂", "雷允上", 
    "南山", "天泰", "东康", "时代", "福仁堂", "龙腾盛世", "存仁堂", "华联", "江南", "心连心", 
    "再康", "昌盛", "康每乐", "开心人", "汇仁堂", "洪兴", "黄庆仁栈", "尧乡", "旭康", "益民", 
    "盛世华兴", "乐仁堂", "崇德", "百和一笑堂", "德医堂", "颈复康", "中亚", "仁泰", "唐人", 
    "华佗", "康仁", "新兴", "狮城百姓", "益友", "石药", "神威", "诚仁堂", "新正", "百姓康", 
    "百草堂", "天宇平价", "医药大厦", "宜生", "神农百草堂", "华为", "伊康", "信锐", "大参林百姓福", 
    "恒生", "百姓健康", "隆泰仁", "天博", "仙鹤", "千年健", "益寿堂", "晟尚", "百氏康", "同心堂", 
    "大参林健康", "林州大药房", "瑞康", "一心堂康健", "东宸", "东森", "丽康", "仁爱健民", "佐今明", 
    "叮当智慧", "同和堂", "名药师", "大中元", "大参林", "天伦", "天康仁", "宜致", "张仲景", "恒信", 
    "普生源", "杏一", "永乐", "润禾", "百事康健", "百家好一生", "中圆", "传仁堂", "华泰宜和堂", 
    "发扬", "国信", "康丰", "康辉", "新越人", "花城", "美锐", "隆祥", "鲲鹏", "一杆秤", "宝神鹿", 
    "心连心", "华辉", "华隆仁康", "晟和", "柏海宏济", "爱心", "蓝十字", "保元堂", "新东方", 
    "胖东来", "九恒", "新向民", "禧康", "德良方", "福郎中", "保济新", "正大", "老百姓", "百姓缘", 
    "大德", "四明", "正源", "彩虹", "易心堂", "为诚人家", "九洲", "华东", "武林", "百合楚济堂", 
    "益万家", "邻家", "丽水便民", "健一行", "华通", "友禾", "国药", "大丛林", "天天好", "张济堂", 
    "惠仁", "振和", "洪福堂", "海港", "华圣", "统一药房", "诚心", "长相宜", "长红", "震元", 
    "一正", "修正堂", "宁康", "布衣", "延生堂", "张和堂", "长生堂", "平家健康", "华枫", "里肯", 
    "英特一洲", "养生堂", "太和堂", "尖峰", "惠民", "民众", "广药晨菲", "鑫九州", "国康", 
    "汉口大药房", "天济", "元昌", "为民天寿", "健康人", "叶开泰", "同仁美康", "聚荣璟", "佳源堂", 
    "惠好", "隆泰益丰", "马应龙", "万众森林", "吴都", "好药师", "孝药", "宏泰", "宜草堂", "寿延堂", 
    "心连心", "晓琳", "普恩堂", "柴氏荣盛", "用心人", "百佳和", "益生天济", "济公", "高济明联", 
    "黎民", "福元堂", "普天独活", "康泰", "神州", "远志", "养天和", "健康南门", "楚济堂", "丹桂园", 
    "九芝堂", "千金", "恒康", "津湘", "益汉", "诚益信", "诺舟", "达嘉维康", "雅馨", "佛慈", 
    "新生", "德生堂", "亚欣", "众友", "心连心", "快康", "至仁同济", "祝强", "鹭燕", "康佰家", 
    "东南", "国大", "榕参", "宜又佳", "恒生", "惠百姓", "新永惠", "永惠", "海华", "康利达", 
    "扬祖惠民", "民心", "燕煌", "百泰", "聚善堂", "聚芝林", "一品", "一树", "正和祥", "健一生", 
    "老天祥", "振安", "百盛新药特药", "泽康", "大厦", "安东新特", "世纪", "亚安", "新立群", 
    "阳光", "敖东", "漱玉", "康源", "东北", "利安德", "九宇康展", "博爱", "康芝悦", "福聚和", 
    "百年东和堂", "万家康", "五洲通", "人民康泰", "北药家", "华诺", "博大维康", "嘉和", "天一优市美", 
    "天士力", "天益堂", "巨力", "康益堂", "建联", "心向民", "成大方圆", "新益康", "施福堂", 
    "旺福营盘", "沛芝堂", "一明", "百和堂", "太星", "福缘堂", "襄元堂", "雪松", "安康", "益春", 
    "弘大惠诚", "华星堂", "万鑫", "和平", "唐氏", "万和", "万家燕", "桐君阁", "泉源堂", "润药渝康", 
    "西部医药商城", "鑫斛", "健生", "中和堂", "同心", "广济堂", "派林", "德厚丰", "怡康", "欣康", 
    "泰生", "东亚", "乐榕融", "乡亲", "京兆", "众信", "五星", "孙思邈", "博爱", "百姓乐", "康德", 
    "朋成", "百家惠", "美致惠", "广济", "长健", "桢州", "九康", "新富康", "万益", "新绿洲", 
    "华伟", "七星堂", "人民同泰", "北秀", "宏腾", "宝丰", "建国", "海晖", "苮丰", "灵峰", 
    "福斯特", "三元至盛", "鑫世一", "利达", "天利", "一辰", "华辰", "安李泰", "大众平安", "安泰", "齐泰","佳康君安"
]

    
    print("KA店铺筛选程序")
    print("=" * 50)
    print(f"候选店铺数量: {len(candidate_stores)} 个")
    
    # 输入文件路径
    input_file_path = input("请输入Excel文件路径: ").strip()
    
    # 检查文件是否存在
    if not os.path.exists(input_file_path):
        print(f"错误: 文件 '{input_file_path}' 不存在")
        return
    
    # 显示文件信息
    print(f"\n文件信息:")
    print(f"  文件路径: {input_file_path}")
    print(f"  文件大小: {os.path.getsize(input_file_path) / 1024:.2f} KB")
    
    # 显示文件中的sheet页
    try:
        xl = pd.ExcelFile(input_file_path)
        sheet_names = xl.sheet_names
        print(f"  可用Sheet页: {sheet_names}")
        
        # 如果只有一个sheet，自动选择
        if len(sheet_names) == 1:
            sheet_name = sheet_names[0]
            print(f"  自动选择Sheet页: {sheet_name}")
        else:
            sheet_name = input("请输入要筛选的Sheet页名称: ").strip()
            
            # 验证sheet名称
            if sheet_name not in sheet_names:
                print(f"错误: Sheet页 '{sheet_name}' 不存在")
                print(f"可用的Sheet页: {sheet_names}")
                return
                
    except Exception as e:
        print(f"读取文件信息失败: {e}")
        return
    
    print(f"\n候选店铺列表:")
    for i, store in enumerate(candidate_stores, 1):
        print(f"  {i:2d}. {store}")
    
    print("\n" + "=" * 50)
    
    # 执行筛选
    output_file = filter_ka_stores(input_file_path, sheet_name, candidate_stores)
    
    if output_file:
        print(f"\n" + "=" * 50)
        print(f"输出文件: {output_file}")
        
        # 显示输出文件大小
        if os.path.exists(output_file):
            file_size = os.path.getsize(output_file) / 1024
            print(f"输出文件大小: {file_size:.2f} KB")
        
        print("筛选完成！")

if __name__ == "__main__":
    main()

KA店铺筛选程序
候选店铺数量: 755 个
请输入Excel文件路径: D:\work\京东优思明0417_去重后_无海外_sorted.xlsx

文件信息:
  文件路径: D:\work\京东优思明0417_去重后_无海外_sorted.xlsx
  文件大小: 114.98 KB
  可用Sheet页: ['Sheet1']
  自动选择Sheet页: Sheet1

候选店铺列表:
   1. 养和堂
   2. 国大
   3. 华氏
   4. 好药师
   5. 崇明第一医药
   6. 第一医药商店
   7. 得一大药房
   8. 海王星辰
   9. 益丰
  10. 雷允上
  11. 一心堂
  12. 老百姓
  13. 东飞药业
  14. 人民大药房
  15. 为了你健康药房
  16. 昊邦
  17. 白药大药房
  18. 百姓仁药业
  19. 医药新特药
  20. 玉溪医药
  21. 立之康
  22. 靓桐
  23. 健之佳
  24. 淞茂
  25. 滇西
  26. 益尔健
  27. 得云药业
  28. 新特新
  29. 仁和堂
  30. 北域健康
  31. 厚道医药
  32. 康中福
  33. 德医堂
  34. 方圆
  35. 惠生
  36. 泽强
  37. 达明堂
  38. 东升天保堂
  39. 日月七天
  40. 金象
  41. 东辽医药
  42. 福春永
  43. 保康
  44. 江城大药房
  45. 福聚康
  46. 敖东
  47. 一笑堂
  48. 万家星火
  49. 义立方
  50. 光明医药
  51. 养天和康赛
  52. 华安大药房
  53. 同仁大药房
  54. 天诺
  55. 恒爱
  56. 房联
  57. 新药特药
  58. 春萍
  59. 永新堂
  60. 百姓润
  61. 百年红星
  62. 禹成
  63. 邱健安康
  64. 金牛大药房
  65. 骅禧
  66. 神农
  67. 益和
  68. 中联保健
  69. 天一堂
  70. 正道
  71. 时珍
  72. 恒爱
  73. 光明万家
  74. 万和堂
  75. 信康
  76. 汇丰
  77. 长新
  78. 义善堂
 

成功读取Sheet页: Sheet1
原始数据行数: 1659
筛选后数据行数: 427
筛选完成！结果已保存到: D:\work\京东优思明0417_去重后_无海外_sorted_Sheet1_KA.xlsx

筛选结果信息:
  总行数: 427
  总列数: 9

按候选店铺统计匹配数量:
  国大: 18 行
  好药师: 1 行
  海王星辰: 17 行
  益丰: 4 行
  老百姓: 18 行
  人民大药房: 4 行
  靓桐: 3 行
  健之佳: 24 行
  惠生: 3 行
  华安大药房: 1 行
  同仁大药房: 1 行
  恒爱: 5 行
  正道: 1 行
  时珍: 4 行
  恒爱: 5 行
  杏林: 3 行
  万盛: 5 行
  百欣堂: 3 行
  健尔堂: 1 行
  百信: 8 行
  惠仁堂: 1 行
  丰原: 12 行
  国胜: 5 行
  达嘉维康: 2 行
  韵天: 4 行
  人民大药房: 4 行
  江南: 3 行
  联众: 1 行
  幸福人: 1 行
  平嘉: 1 行
  众康: 1 行
  漱玉: 4 行
  中医世家: 7 行
  惠仁: 1 行
  健民: 1 行
  益民: 8 行
  天诚: 1 行
  中智: 1 行
  佛心: 6 行
  五福堂: 4 行
  好万家: 8 行
  新合健: 5 行
  爱心: 10 行
  益康: 1 行
  东兴堂: 7 行
  健民: 1 行
  广药大药房: 8 行
  广药: 12 行
  百姓缘: 1 行
  益民: 8 行
  健春堂: 1 行
  桂中: 2 行
  康之源: 3 行
  惠生堂: 3 行
  新特: 1 行
  康宁: 1 行
  上元堂: 2 行
  百信: 8 行
  京典: 2 行
  普泽: 1 行
  济生堂: 1 行
  恩华统一: 4 行
  九州: 1 行
  同慈: 2 行
  众药师: 1 行
  国瑞堂: 1 行
  舒健: 1 行
  兴药: 1 行
  德新元: 3 行
  芝林: 6 行
  济生: 1 行
  华联: 3 行
  江南: 3 行
  昌盛: 4 行
  开心人: 3 行
  汇仁堂: 6 行
  益民: 8 行
  唐人: 4 行
  康仁: 1 行
  新兴: 2 行

In [3]:
import pandas as pd
import os
import re
from datetime import datetime

class KAStoreFilter:
    def __init__(self):
        self.candidate_stores = [
            "海王星辰", "益丰", "老百姓", "健之佳",
            "漱玉", "广药", "大参林", "国大","佳康君安"
        ]
        # 明确指定价格列名称
        self.price_column = '到手_优惠价_清洗后'
        
    def get_store_group(self, store_name):
        """根据店铺名称获取店铺分组（店铺名称中出现相同关键词的视为一家店）"""
        if pd.isna(store_name):
            return None
            
        store_name_str = str(store_name).lower()
        for store_keyword in self.candidate_stores:
            if store_keyword.lower() in store_name_str:
                return store_keyword
        return None
    
    def filter_by_product_pattern(self, df, product_pattern):

     if product_pattern == '21片':
        # 筛选21/片和1盒装，且不包含"+"和"63片"
        def contains_both_patterns_21(text):
            if pd.isna(text):
                return False
            text_str = str(text)
            
            # 如果包含"+"或"63片", 直接返回False
            if '+' in text_str:
                return False
            if '63片' in text_str:
                return False
            
            has_21_per_piece = re.search(r'21\s*[/／]\s*片|21\s*片|21/片|21／片', text_str, re.IGNORECASE) is not None
            has_1_box = re.search(r'1\s*盒\s*装|1\s*盒|1盒装|1盒', text_str, re.IGNORECASE) is not None
            return has_21_per_piece and has_1_box
        
        condition = df['商品名称'].apply(contains_both_patterns_21)
        
     elif product_pattern == '28片':
        # 筛选28/片和1盒装，且不包含"+"和"63片"
        def contains_both_patterns_28(text):
            if pd.isna(text):
                return False
            text_str = str(text)
            
            # 如果包含"+"或"63片", 直接返回False
            if '+' in text_str:
                return False
            if '63片' in text_str:
                return False
            
            has_28_per_piece = re.search(r'28\s*[/／]\s*片|28\s*片|28/片|28／片', text_str, re.IGNORECASE) is not None
            has_1_box = re.search(r'1\s*盒\s*装|1\s*盒|1盒装|1盒', text_str, re.IGNORECASE) is not None
            return has_28_per_piece and has_1_box
        
        condition = df['商品名称'].apply(contains_both_patterns_28)
    
     return df[condition]
    


    

    

    
    def find_lowest_price_per_store(self, df):
        """
        找出每家店的最低价格商品记录
        
        参数:
        df: 筛选后的数据
        
        返回:
        每家店最低价格记录的数据
        """
        if len(df) == 0:
            return pd.DataFrame()
        
        # 检查是否有指定的价格列
        if self.price_column not in df.columns:
            print(f"⚠️  警告: 未找到价格列 '{self.price_column}'")
            # 尝试寻找其他可能的价格列
            price_columns = ['价格', '单价', '售价', '成交价', '优惠价', '到手价']
            alternative_price_column = None
            for col in price_columns:
                if col in df.columns:
                    alternative_price_column = col
                    print(f"  找到替代价格列: '{col}'")
                    break
            
            if alternative_price_column is None:
                print("  未找到任何价格列，无法计算最低价格")
                return pd.DataFrame()
            else:
                price_column = alternative_price_column
                print(f"  将使用替代价格列: '{price_column}'")
        else:
            price_column = self.price_column
            print(f"  使用指定价格列: '{price_column}'")
        
        # 添加店铺分组列
        df_with_group = df.copy()
        df_with_group['店铺分组'] = df_with_group['店铺名称'].apply(self.get_store_group)
        
        # 删除无法分组的行
        df_with_group = df_with_group[df_with_group['店铺分组'].notna()]
        
        # 转换价格为数值类型
        try:
            # 先移除可能的非数字字符（如货币符号、逗号等）
            df_with_group[price_column] = df_with_group[price_column].astype(str).str.replace('[^\d.]', '', regex=True)
            df_with_group[price_column] = pd.to_numeric(df_with_group[price_column], errors='coerce')
            print(f"  价格列转换完成，有效价格记录数: {df_with_group[price_column].notna().sum()}")
        except Exception as e:
            print(f"  价格转换错误: {e}")
            return pd.DataFrame()
        
        # 检查是否有有效的价格数据
        if df_with_group[price_column].isna().all():
            print("  ⚠️ 所有价格数据都是NaN，无法计算最低价格")
            return pd.DataFrame()
        
        # 按店铺分组找出最低价格记录
        lowest_price_records = []
        found_stores = []
        
        for store_keyword in self.candidate_stores:
            store_data = df_with_group[df_with_group['店铺分组'] == store_keyword]
            
            if len(store_data) > 0:
                # 找出非NaN的最低价格
                store_data_valid = store_data[store_data[price_column].notna()]
                
                if len(store_data_valid) > 0:
                    min_price = store_data_valid[price_column].min()
                    
                    # 获取所有最低价格的记录
                    min_price_records = store_data_valid[store_data_valid[price_column] == min_price]
                    
                    # 如果有多条相同最低价格的记录，取第一条
                    if len(min_price_records) > 0:
                        lowest_price_record = min_price_records.iloc[0]
                        lowest_price_records.append(lowest_price_record)
                        found_stores.append(store_keyword)
                        print(f"    • {store_keyword}: 最低价格 = {min_price}")
        
        print(f"  找到 {len(found_stores)} 家店的最低价格记录")
        
        if lowest_price_records:
            result_df = pd.DataFrame(lowest_price_records)
            # 按店铺分组排序
            result_df = result_df.sort_values('店铺分组')
            return result_df
        else:
            return pd.DataFrame()
    
    def process_file(self, input_file_path, sheet_name):
        """
        处理Excel文件
        
        参数:
        input_file_path: 输入文件路径
        sheet_name: 要读取的sheet名称
        
        返回:
        (success, message, output_path, filtered_dfs)
        """
        print(f"正在读取Sheet页: {sheet_name}...")
        try:
            df = pd.read_excel(input_file_path, sheet_name=sheet_name)
            print(f"✓ 成功读取Sheet页: {sheet_name}，共 {len(df)} 行数据")
        except Exception as e:
            print(f"✗ 读取文件失败: {e}")
            return False, f"读取文件失败: {e}", None, None
        
        # 检查必要的列是否存在
        required_columns = ['店铺名称', '商品名称']
        missing_columns = [col for col in required_columns if col not in df.columns]
        
        if missing_columns:
            error_msg = f"Excel文件中没有找到以下列: {', '.join(missing_columns)}"
            print(f"✗ 错误: {error_msg}")
            print(f"可用列: {', '.join(list(df.columns))}")
            return False, error_msg, None, None
        
        # 检查价格列是否存在
        if self.price_column not in df.columns:
            print(f"⚠️ 注意: 未找到指定价格列 '{self.price_column}'")
            print("将尝试寻找其他可能的价格列...")
            
        # 第一步：筛选候选店铺
        print(f"\n第一步：筛选候选店铺，候选店铺数: {len(self.candidate_stores)}...")
        original_count = len(df)
        
        store_condition = pd.Series(False, index=df.index)
        for store in self.candidate_stores:
            store_condition = store_condition | df['店铺名称'].astype(str).str.contains(store, case=False, na=False)
        
        store_filtered_df = df[store_condition]
        store_filtered_count = len(store_filtered_df)
        
        print(f"原始数据行数: {original_count}")
        print(f"店铺筛选后数据行数: {store_filtered_count}")
        print(f"店铺筛选比例: {store_filtered_count/original_count:.2%}")
        
        if store_filtered_count == 0:
            print("⚠️  警告: 没有筛选到任何店铺数据！")
            return False, "没有筛选到任何店铺数据", None, None
        
        # 第二步：分别筛选优思明和优思悦商品
        print(f"\n第二步：筛选商品数据...")
        
        # 筛选优思明商品（21/片 + 1盒装）
        print("  - 筛选优思明商品（21/片 + 1盒装）...")
        filtered_21_df = self.filter_by_product_pattern(store_filtered_df, '21片')
        filtered_21_count = len(filtered_21_df)
        print(f"    筛选到 {filtered_21_count} 条记录")
        
        # 筛选优思悦商品（28/片 + 1盒装）
        print("  - 筛选优思悦商品（28/片 + 1盒装）...")
        filtered_28_df = self.filter_by_product_pattern(store_filtered_df, '28片')
        filtered_28_count = len(filtered_28_df)
        print(f"    筛选到 {filtered_28_count} 条记录")
        
        print(f"\n📊 商品筛选统计:")
        print(f"  • 优思明商品（21片）: {filtered_21_count} 条")
        print(f"  • 优思悦商品（28片）: {filtered_28_count} 条")
        
        # 显示前几个商品名称示例
        if filtered_21_count > 0:
            print(f"\n  优思明商品示例:")
            for i, product in enumerate(filtered_21_df['商品名称'].head(3), 1):
                print(f"    {i}. {str(product)[:80]}...")
        
        if filtered_28_count > 0:
            print(f"\n  优思悦商品示例:")
            for i, product in enumerate(filtered_28_df['商品名称'].head(3), 1):
                print(f"    {i}. {str(product)[:80]}...")
        
        # 第三步：找出每家店的最低价格商品记录
        print(f"\n第三步：找出每家店的最低价格商品记录...")
        
        # 优思明最低价格记录
        print("  - 处理优思明最低价格记录...")
        lowest_price_21_df = self.find_lowest_price_per_store(filtered_21_df)
        lowest_price_21_count = len(lowest_price_21_df)
        print(f"    找到 {lowest_price_21_count} 家店的优思明最低价格记录")
        
        # 优思悦最低价格记录
        print("  - 处理优思悦最低价格记录...")
        lowest_price_28_df = self.find_lowest_price_per_store(filtered_28_df)
        lowest_price_28_count = len(lowest_price_28_df)
        print(f"    找到 {lowest_price_28_count} 家店的优思悦最低价格记录")
        
        # 准备输出数据
        filtered_dfs = {
            '优思明筛选结果': filtered_21_df,
            '优思悦筛选结果': filtered_28_df,
            '优思明最低价格': lowest_price_21_df,
            '优思悦最低价格': lowest_price_28_df
        }
        
        return True, "处理完成", None, filtered_dfs
    
    def save_results(self, input_file_path, sheet_name, filtered_dfs):
        """保存结果到Excel文件"""
        if not filtered_dfs:
            return None
        
        # 生成输出文件名
        file_dir = os.path.dirname(input_file_path)
        file_name = os.path.basename(input_file_path)
        name_part, ext = os.path.splitext(file_name)
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file_name = f"{name_part}_{sheet_name}_综合筛选结果_{timestamp}{ext}"
        output_file_path = os.path.join(file_dir, output_file_name)
        
        print(f"\n正在保存文件: {output_file_name}...")
        
        try:
            # 使用ExcelWriter创建多sheet的Excel文件
            with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
                # Sheet1: 优思明筛选结果
                if len(filtered_dfs['优思明筛选结果']) > 0:
                    filtered_dfs['优思明筛选结果'].to_excel(writer, sheet_name='优思明筛选结果', index=False)
                    print(f"  ✓ 已保存Sheet1: 优思明筛选结果 ({len(filtered_dfs['优思明筛选结果'])} 行)")
                else:
                    pd.DataFrame({'说明': ['无符合条件的数据']}).to_excel(writer, sheet_name='优思明筛选结果', index=False)
                    print(f"  ⚠️ Sheet1: 优思明筛选结果 (无数据)")
                
                # Sheet2: 优思悦筛选结果
                if len(filtered_dfs['优思悦筛选结果']) > 0:
                    filtered_dfs['优思悦筛选结果'].to_excel(writer, sheet_name='优思悦筛选结果', index=False)
                    print(f"  ✓ 已保存Sheet2: 优思悦筛选结果 ({len(filtered_dfs['优思悦筛选结果'])} 行)")
                else:
                    pd.DataFrame({'说明': ['无符合条件的数据']}).to_excel(writer, sheet_name='优思悦筛选结果', index=False)
                    print(f"  ⚠️ Sheet2: 优思悦筛选结果 (无数据)")
                
                # Sheet3: 优思明最低价格
                if len(filtered_dfs['优思明最低价格']) > 0:
                    # 确定价格列名称
                    price_columns_in_df = [col for col in [self.price_column, '价格', '单价', '售价', '成交价'] 
                                         if col in filtered_dfs['优思明最低价格'].columns]
                    price_column = price_columns_in_df[0] if price_columns_in_df else '价格'
                    
                    filtered_dfs['优思明最低价格'].to_excel(writer, sheet_name='优思明最低价格', index=False)
                    print(f"  ✓ 已保存Sheet3: 优思明最低价格 ({len(filtered_dfs['优思明最低价格'])} 行)")
                    
                    # 显示优思明最低价格信息
                    if price_column in filtered_dfs['优思明最低价格'].columns:
                        print(f"\n  📊 优思明最低价格统计:")
                        for store in self.candidate_stores:
                            store_data = filtered_dfs['优思明最低价格'][filtered_dfs['优思明最低价格']['店铺分组'] == store]
                            if len(store_data) > 0:
                                price = store_data.iloc[0][price_column]
                                print(f"    • {store}: {price}")
                else:
                    pd.DataFrame({'说明': ['无符合条件的数据']}).to_excel(writer, sheet_name='优思明最低价格', index=False)
                    print(f"  ⚠️ Sheet3: 优思明最低价格 (无数据)")
                
                # Sheet4: 优思悦最低价格
                if len(filtered_dfs['优思悦最低价格']) > 0:
                    # 确定价格列名称
                    price_columns_in_df = [col for col in [self.price_column, '价格', '单价', '售价', '成交价'] 
                                         if col in filtered_dfs['优思悦最低价格'].columns]
                    price_column = price_columns_in_df[0] if price_columns_in_df else '价格'
                    
                    filtered_dfs['优思悦最低价格'].to_excel(writer, sheet_name='优思悦最低价格', index=False)
                    print(f"  ✓ 已保存Sheet4: 优思悦最低价格 ({len(filtered_dfs['优思悦最低价格'])} 行)")
                    
                    # 显示优思悦最低价格信息
                    if price_column in filtered_dfs['优思悦最低价格'].columns:
                        print(f"\n  📊 优思悦最低价格统计:")
                        for store in self.candidate_stores:
                            store_data = filtered_dfs['优思悦最低价格'][filtered_dfs['优思悦最低价格']['店铺分组'] == store]
                            if len(store_data) > 0:
                                price = store_data.iloc[0][price_column]
                                print(f"    • {store}: {price}")
                else:
                    pd.DataFrame({'说明': ['无符合条件的数据']}).to_excel(writer, sheet_name='优思悦最低价格', index=False)
                    print(f"  ⚠️ Sheet4: 优思悦最低价格 (无数据)")
            
            print(f"\n✓ 筛选完成！结果已保存到: {output_file_path}")
            
            # 显示文件大小
            if os.path.exists(output_file_path):
                file_size = os.path.getsize(output_file_path) / 1024
                print(f"📊 输出文件大小: {file_size:.2f} KB")
            
            return output_file_path
            
        except Exception as e:
            print(f"✗ 保存文件失败: {e}")
            return None

def main():
    # 创建筛选器实例
    filter_tool = KAStoreFilter()
    
    print("=" * 60)
    print("             综合KA店铺筛选程序")
    print("=" * 60)
    print("功能说明:")
    print("  1. 筛选优思明商品（21/片 + 1盒装）")
    print("  2. 筛选优思悦商品（28/片 + 1盒装）")
    print("  3. 找出每家店的最低价格商品记录")
    print(f"  价格列: '{filter_tool.price_column}'")
    print("=" * 60)
    print(f"候选店铺数量: {len(filter_tool.candidate_stores)} 个")
    print("店铺列表:")
    for i, store in enumerate(filter_tool.candidate_stores, 1):
        print(f"  {i:2d}. {store}")
    print("=" * 60)
    
    # 输入文件路径
    input_file_path = input("\n📂 请输入Excel文件路径: ").strip()
    
    # 检查文件是否存在
    if not os.path.exists(input_file_path):
        print(f"✗ 错误: 文件 '{input_file_path}' 不存在")
        return
    
    # 显示文件信息
    print(f"\n📋 文件信息:")
    print(f"  • 文件路径: {input_file_path}")
    print(f"  • 文件大小: {os.path.getsize(input_file_path) / 1024:.2f} KB")
    
    # 显示文件中的sheet页
    try:
        xl = pd.ExcelFile(input_file_path)
        sheet_names = xl.sheet_names
        print(f"  • 可用Sheet页: {len(sheet_names)} 个")
        
        # 显示所有sheet页
        for i, sheet in enumerate(sheet_names, 1):
            print(f"    {i:2d}. {sheet}")
        
        # 如果只有一个sheet，自动选择
        if len(sheet_names) == 1:
            sheet_name = sheet_names[0]
            print(f"\n  • 自动选择Sheet页: {sheet_name}")
        else:
            sheet_name = input("\n📄 请输入要筛选的Sheet页名称: ").strip()
            
            # 验证sheet名称
            if sheet_name not in sheet_names:
                print(f"✗ 错误: Sheet页 '{sheet_name}' 不存在")
                print(f"可用的Sheet页: {', '.join(sheet_names)}")
                return
                
    except Exception as e:
        print(f"✗ 读取文件信息失败: {e}")
        return
    
    print(f"\n🛒 商品筛选条件:")
    print(f"  优思明商品: 包含 '21/片' 和 '1盒装'")
    print(f"  优思悦商品: 包含 '28/片' 和 '1盒装'")
    print(f"  （支持多种写法，不区分大小写）")
    
    print(f"\n💰 最低价格筛选:")
    print(f"  价格列: '{filter_tool.price_column}'")
    print(f"  将从筛选结果中找出8家店铺的最低价格商品记录")
    print(f"  店铺分组规则: 店铺名称中出现相同关键词的视为一家店")
    
    print("\n" + "=" * 60)
    
    # 确认操作
    confirm = input("是否开始筛选? (y/n): ").strip().lower()
    if confirm != 'y':
        print("操作已取消")
        return
    
    # 执行筛选
    print("\n" + "=" * 60)
    print("开始筛选...")
    print("=" * 60)
    
    success, message, _, filtered_dfs = filter_tool.process_file(input_file_path, sheet_name)
    
    if success and filtered_dfs:
        # 保存结果
        output_file = filter_tool.save_results(input_file_path, sheet_name, filtered_dfs)
        
        if output_file:
            print("\n" + "=" * 60)
            print("             筛选完成！")
            print("=" * 60)
            print(f"📁 输出文件: {output_file}")
            print("\n📋 输出文件包含以下Sheet页:")
            print("  1. 优思明筛选结果")
            print("  2. 优思悦筛选结果")
            print("  3. 优思明最低价格（每家店最低价）")
            print("  4. 优思悦最低价格（每家店最低价）")
            print("\n✅ 程序执行完毕！")
            print("=" * 60)
    else:
        print(f"\n✗ 处理失败: {message}")
        print("请检查输入文件和筛选条件。")

if __name__ == "__main__":
    main()

             综合KA店铺筛选程序
功能说明:
  1. 筛选优思明商品（21/片 + 1盒装）
  2. 筛选优思悦商品（28/片 + 1盒装）
  3. 找出每家店的最低价格商品记录
  价格列: '到手_优惠价_清洗后'
候选店铺数量: 9 个
店铺列表:
   1. 海王星辰
   2. 益丰
   3. 老百姓
   4. 健之佳
   5. 漱玉
   6. 广药
   7. 大参林
   8. 国大
   9. 佳康君安

📂 请输入Excel文件路径: D:\work\京东优思明0417_去重后_无海外_sorted_Sheet1_KA.xlsx

📋 文件信息:
  • 文件路径: D:\work\京东优思明0417_去重后_无海外_sorted_Sheet1_KA.xlsx
  • 文件大小: 33.14 KB
  • 可用Sheet页: 1 个
     1. Sheet1

  • 自动选择Sheet页: Sheet1

🛒 商品筛选条件:
  优思明商品: 包含 '21/片' 和 '1盒装'
  优思悦商品: 包含 '28/片' 和 '1盒装'
  （支持多种写法，不区分大小写）

💰 最低价格筛选:
  价格列: '到手_优惠价_清洗后'
  将从筛选结果中找出8家店铺的最低价格商品记录
  店铺分组规则: 店铺名称中出现相同关键词的视为一家店

是否开始筛选? (y/n): y

开始筛选...
正在读取Sheet页: Sheet1...
✓ 成功读取Sheet页: Sheet1，共 427 行数据

第一步：筛选候选店铺，候选店铺数: 9...
原始数据行数: 427
店铺筛选后数据行数: 97
店铺筛选比例: 22.72%

第二步：筛选商品数据...
  - 筛选优思明商品（21/片 + 1盒装）...
    筛选到 24 条记录
  - 筛选优思悦商品（28/片 + 1盒装）...
    筛选到 0 条记录

📊 商品筛选统计:
  • 优思明商品（21片）: 24 条
  • 优思悦商品（28片）: 0 条

  优思明商品示例:
    1. [优思明]屈螺酮炔雌醇片 0.03mg：3mg*21片 1盒装 0.03mg：3...
    2. [优思明]屈螺酮炔雌醇片 0.03mg：3mg*21片 1盒装 屈